In [1]:
import pandas as pd

# Read the Excel file into a DataFrame
file_path = "D:\OVGU\AMS project\GIT\AMS_ShipTraj\physics based model\Rheinhafen Data.xlsx"
df = pd.read_excel(file_path, header=None)
print(df.head())

         0                       1  \
0  3730963 2025-03-09 00:00:00.397   
1  3730964 2025-03-09 00:00:04.230   
2  3730965 2025-03-09 00:00:06.277   
3  3730966 2025-03-09 00:00:07.531   
4  3730967 2025-03-09 00:00:10.537   

                                                   2  
0  {"msg_type": 1, "repeat": 0, "mmsi": 244660361...  
1  {"msg_type": 1, "repeat": 0, "mmsi": 211576360...  
2  {"msg_type": 1, "repeat": 0, "mmsi": 211478290...  
3  {"msg_type": 1, "repeat": 0, "mmsi": 211632780...  
4  {"msg_type": 1, "repeat": 0, "mmsi": 244660361...  


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

# Step 1: Combine strings
df['combined'] = df.astype(str).agg(' '.join, axis=1)

# Step 2: Filter by MMSI
mmsi_id = "244660361"
filtered_df = df[df['combined'].str.contains(f'"mmsi": {mmsi_id}')].copy()

# Step 3: Extract timestamp, lat, lon
def extract_field(pattern, text):
    match = re.search(pattern, text)
    return match.group(1) if match else None

filtered_df['timestamp'] = filtered_df['combined'].apply(lambda x: extract_field(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)', x))
filtered_df['lon'] = filtered_df['combined'].apply(lambda x: extract_field(r'"lon": ([\d\.]+)', x))
filtered_df['lat'] = filtered_df['combined'].apply(lambda x: extract_field(r'"lat": ([\d\.]+)', x))

# Step 4: Convert types
filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
filtered_df['lat'] = filtered_df['lat'].astype(float)
filtered_df['lon'] = filtered_df['lon'].astype(float)

# Step 5: Sort by time
filtered_df = filtered_df.sort_values(by='timestamp').reset_index(drop=True)

# Step 6: Define angle function
def angle_between(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    dot = np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0)
    angle_rad = np.arccos(dot)
    return np.degrees(angle_rad)

# Step 7: Segmenting based on BOTH time gap >= 1 day AND reverse direction
angle_threshold = 120  # Reversal
segments = []
current_segment = [filtered_df.iloc[0]]

for i in range(2, len(filtered_df)):
    p0 = filtered_df.iloc[i - 2]
    p1 = filtered_df.iloc[i - 1]
    p2 = filtered_df.iloc[i]

    # Time gap between p1 and p2
    time_gap = (p2['timestamp'] - p1['timestamp']).total_seconds() / (3600 * 24)  # in days

    # Direction vectors
    v1 = np.array([p1['lon'] - p0['lon'], p1['lat'] - p0['lat']])
    v2 = np.array([p2['lon'] - p1['lon'], p2['lat'] - p1['lat']])
    angle = angle_between(v1, v2)

    if time_gap >= 1 and angle > angle_threshold:
        segments.append(current_segment)
        current_segment = [p2]
    else:
        current_segment.append(p2)

# Final segment
if current_segment:
    segments.append(current_segment)




In [9]:
print(len(filtered_df))
print(filtered_df['lat'][0],filtered_df['lon'][0])
print(filtered_df['lat'][len(filtered_df)-1],filtered_df['lon'][len(filtered_df)-1])


271
48.981312 8.262617
49.098138 8.359107


In [3]:
import folium

# Center the map at the first coordinate (non-NaN)
start_lat = filtered_df['lat'].dropna().iloc[0]
start_lon = filtered_df['lon'].dropna().iloc[0]
m = folium.Map(location=[start_lat, start_lon], zoom_start=13)

# Add each segment as a polyline
for seg in segments:
    seg_df = pd.DataFrame(seg)
    
    # Drop rows with NaN values before plotting
    seg_df = seg_df.dropna(subset=['lat', 'lon'])
    
    if seg_df.empty:
        continue
    
    coords = seg_df[['lat', 'lon']].values.tolist()
    folium.PolyLine(coords, color='blue', weight=3).add_to(m)

# Add start marker (red)
folium.Marker(
    location=[start_lat, start_lon],
    tooltip="Start",
    icon=folium.Icon(color='red', icon='play')
).add_to(m)

# Add end marker (green)
last_seg = pd.DataFrame(segments[-1]).dropna(subset=['lat', 'lon'])
if not last_seg.empty:
    folium.Marker(
        location=[last_seg.iloc[-1]['lat'], last_seg.iloc[-1]['lon']],
        tooltip="End",
        icon=folium.Icon(color='green', icon='stop')
    ).add_to(m)

# Display map in notebook
m


In [4]:
import numpy as np
import folium

def draw_smooth_entry_to_ais(start, df):
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))

    def bezier(t, p0, p1, p2, p3):
        return (1-t)**3*p0 + 3*(1-t)**2*t*p1 + 3*(1-t)*t**2*p2 + t**3*p3

    df = df.sort_values(by="timestamp").reset_index(drop=True)

    # Find closest AIS point
    distances = df.apply(lambda row: haversine(start["lat"], start["lon"], row["lat"], row["lon"]), axis=1)
    idx = distances.idxmin()
    end_row = df.loc[idx]
    end_lat, end_lon = end_row["lat"], end_row["lon"]
    p3 = np.array([end_lon, end_lat])
    p0 = np.array([start["lon"], start["lat"]])

    # Estimate tangent at AIS point using previous or next point
    if idx > 0:
        ref_row = df.loc[idx - 1]
    elif idx < len(df) - 1:
        ref_row = df.loc[idx + 1]
    else:
        raise ValueError("Not enough AIS points to estimate direction.")

    direction = np.array([end_lon - ref_row["lon"], end_lat - ref_row["lat"]])
    direction = direction / np.linalg.norm(direction + 1e-9)  # avoid divide by zero
    scale = 0.01  # you can tune this
    p2 = p3 - scale * direction

    # Simple inward control point from start (not directional)
    p1 = p0 + 0.01 * (p3 - p0)

    # Create curve
    t_vals = np.linspace(0, 1, 50)
    curve = np.array([bezier(t, p0, p1, p2, p3) for t in t_vals])
    curve_latlon = [[lat, lon] for lon, lat in curve]

    # Remaining AIS path
    continuation_path = df.loc[idx:][['lat', 'lon']].dropna().values.tolist()

    # Create folium map
    m = folium.Map(location=[start["lat"], start["lon"]], zoom_start=14)
    folium.PolyLine(locations=curve_latlon, color='orange', weight=4, tooltip="Smooth Entry").add_to(m)
    folium.PolyLine(locations=continuation_path, color='blue', weight=3, tooltip="AIS Path").add_to(m)

    # Markers
    folium.Marker([start["lat"], start["lon"]], tooltip="Start Point", icon=folium.Icon(color="red")).add_to(m)
    folium.Marker([end_lat, end_lon], tooltip="Join to AIS", icon=folium.Icon(color="green")).add_to(m)

    return m
